# Cohere Parse Models in SageMaker  


In this notebook, we demonstrate how to use the SageMaker Python SDK to deploy and run inference on Parse models from Cohere. Parse is a vision language model that converts unstructured data from enterprise documents (PDFs, images, slides) into structured Markdown output.

## Setup


First, upgrade to the latest Sagemaker, Boto and Cohere SDKs to be able to deploy and invoke Cohere models.

In [ ]:
%pip install --upgrade --quiet sagemaker boto3 cohere

## Deploy

#### Available Models
| **Model Name**        | **Model ID**                       |
|----------------------|------------------------------------|
| Parse v5.0 | `cohere-parse-v5` |

Specify the model ID to deploy one of the models from the above list.

In [ ]:
model_id, model_version = "cohere-parse-v5", "*"

Use the Sagemaker Jumpstart SDK to deploy the model.

In [ ]:
from sagemaker.jumpstart.model import JumpStartModel

model = JumpStartModel(model_id=model_id, model_version=model_version)
deployed_model = model.deploy()

# Use the model

### Parse a document image

The Parse API accepts a document as an image URL (including base64-encoded data URIs) and returns structured Markdown output.

In [ ]:
import base64
from pathlib import Path


def image_to_data_uri(image_path: str) -> str:
    """Convert a local image file to a base64-encoded data URI."""
    path = Path(image_path)
    suffix = path.suffix.lower().lstrip(".")
    mime_type = {
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
        "png": "image/png",
        "gif": "image/gif",
        "webp": "image/webp",
    }.get(suffix, "image/png")

    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    return f"data:{mime_type};base64,{encoded}"

#### Parse a single image

Replace `"your_document.png"` with the path to your document image or PDF page.

In [ ]:
import cohere

co = cohere.SagemakerClientV2()

# Convert your document image to a data URI
image_path = "your_document.png"  # Replace with your image path
data_uri = image_to_data_uri(image_path)

response = co.parse(
    model=deployed_model.endpoint_name,
    document={"type": "image_url", "image_url": data_uri},
)

# Print the parsed Markdown content
for page in response.pages:
    print(page.markdown.content)

#### Parse with blocks output

You can request blocks output which returns structured content blocks (text, tables, figures, etc.) with bounding box coordinates.

In [ ]:
response = co.parse(
    model=deployed_model.endpoint_name,
    document={"type": "image_url", "image_url": data_uri},
    output_format="blocks",
)

for page in response.pages:
    for block in page.blocks:
        if block.type == "text":
            print(block.text.content)
        elif block.type == "table":
            print(f"[Table] bbox={block.table.bounding_box}")
            print(block.table.html)
            print(block.table.description)
        print()

### More API Features
To learn more about the Parse model and its capabilities, see the [Cohere Parse documentation](https://docs.cohere.com/docs/parse).

# Clean Up

After using the resource, you can delete the model and the endpoint.

In [ ]:
deployed_model.delete_model()
deployed_model.delete_endpoint()